# VVRAG ingestion (free GPU)
Self-contained: run top to bottom, no setup needed. Clones the repo, installs deps, probes the environment (retriever Option A check), ingests, zips `data/index` for download. No logic lives here.

In [ ]:
!test -d altVRAG || git clone https://github.com/BiprashPandey/altVRAG.git
%cd altVRAG
!pip install -q -e . --no-deps
!mkdir -p data/raw && ls data/raw

Upload your PDFs/slide images into the `altVRAG/data/raw` folder now (Files panel → right-click → Upload), then continue. `data/raw/` exists right after cloning (committed skeleton).

In [ ]:
%pip install -q colpali-engine qdrant-client pymupdf pillow pytesseract transformers torch --index-url https://pypi.org/simple
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!python -c "from vvrag.ingest.embed import probe_environment; probe_environment()"

## 1. Verify the retriever checkpoint (Option A)
Check that `vidore/colqwen2-v1.0` resolves on the Hub. If it fails (gated/renamed), log in (`!huggingface-cli login`) or pick a working ColPali checkpoint and pass `--model-id` in cell 3.

In [ ]:
!huggingface-cli download vidore/colqwen2-v1.0 --dry-run 2>&1 | head -5
# If the line above errors, set MODEL_ID below, e.g. vidore/colpali-v1.2

In [ ]:
!ls data/raw
!python -m vvrag.ingest.run_ingest --input data/raw --out data/index --embedder colqwen --limit 3
# Smoke test ingests 3 files. If manifest looks right, re-run without --limit for the full corpus.

In [ ]:
!cat data/index/manifest.json
!zip -qr index.zip data/index && ls -lh index.zip
# Download index.zip (Files panel) and unzip it into data/index/ on your laptop.